In [1]:
import pandas as pd
import string
from sklearn.model_selection import train_test_split
from numpy.lib.stride_tricks import sliding_window_view
import matplotlib.pyplot as plt
import random
import numpy as np
import math
dataset = pd.read_csv("sign_lang_mnist/sign_mnist_train/sign_mnist_train.csv")
alphabet = {index:letter for index, letter in enumerate(string.ascii_lowercase)} 
prex = dataset.drop(columns=["label"])
X = prex.to_numpy().reshape(-1,1,28,28)
y = dataset["label"].to_numpy()
print(y[3])
X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
    )

# sow = [random.randint(0,len(X_train)-1) for i in range(10)]

# for i in sow:
#     img = X_train[i]
#     label = y_train[i]
#     plt.figure(figsize=(50,50))

#     plt.subplot(28, 28, 1)        
#     plt.imshow(img, cmap=plt.cm.gray)
#     plt.title(f"alphabet {alphabet.get(label,'unknown')}")

2


In [2]:
max_pooled_activations = []
activations = []
pre_activations = []
beta_1 = 0.90
beta_2 = 0.999
learning_rate = 0.001
moment_1_history = {}
moment_2_history = {}

bias_moment_1_history = {}
bias_moment_2_history = {}

epsilon = 1e-8

conv2_flattened_activatons = None

In [3]:
def softmax(pre_activations):
    # axis=-1 safely handles both 1D vectors and 2D batches
    # Subtracting in-place or inline saves memory overhead
    exps = np.exp(pre_activations - np.max(pre_activations, axis=-1, keepdims=True))
    
    return exps / np.sum(exps, axis=-1, keepdims=True)

In [4]:
def sparse_categorical_crossentropy_loss(predictions, true_classes):
    eps = 1e-15
    preds  = np.clip(predictions,eps,1-eps)
    indexes = true_classes.index
    loss = -np.mean(np.log(preds[np.arange(len(true_classes)),true_classes]))
    return loss


In [5]:
def accuracy(batch_predictions, true_classes):
    preds = np.asarray(batch_predictions)
    targets = np.asarray(true_classes)
    
    predicted_classes = np.argmax(preds, axis=1)
    matches = (predicted_classes ==  targets)
    accuracy = np.mean(matches)

    print(f"Predicted: {predicted_classes}")
    print(f"Matches:   {matches}")
    print(f"Accuracy:  {accuracy:.2%}")
    

In [6]:
def softmax_sparse_categorical_crossentropy_gradient(logits_prediction_arr, true_class_idx):
    grad = logits_prediction_arr.copy()
    for i in range(len(logits_prediction_arr)):
        grad[i][true_class_idx[i]]-=1
    return grad

In [7]:
def relu(bias_added_weights):
    return np.where(bias_added_weights>0,bias_added_weights,0)

In [8]:
def relu_derivative(pre_activations):
    return (pre_activations>0).astype(float)

In [9]:
def adam_optimizer(curr_weights, curr_gradients, iter, layer, is_bias=False):

    if is_bias:
        moment_1_curr = (beta_1*(bias_moment_1_history.get(layer,np.zeros(curr_weights.shape)))) + (1-beta_1) * curr_gradients
        bias_moment_1_history[layer] = moment_1_curr
    else:
        moment_1_curr = (beta_1*(moment_1_history.get(layer,np.zeros(curr_weights.shape)))) + (1-beta_1) * curr_gradients
        moment_1_history[layer] = moment_1_curr

    hat_moment_1 = moment_1_curr/(1-beta_1**iter)

    if is_bias:
        moment_2_curr = beta_2*(bias_moment_2_history.get(layer,np.zeros(curr_weights.shape))) + (1-beta_2) * curr_gradients**2
        bias_moment_2_history[layer] = moment_2_curr
    else:
        moment_2_curr = beta_2*(moment_2_history.get(layer,np.zeros(curr_weights.shape))) + (1-beta_2) * curr_gradients**2
        moment_2_history[layer] = moment_2_curr

    hat_moment_2 = moment_2_curr/(1-beta_2**iter)

    return ((learning_rate/(np.sqrt(hat_moment_2)+epsilon)) * hat_moment_1)

In [10]:
def max_pooling(activations_matrices):
    stride_row,strice_col=(2,2)
    slided_pool = sliding_window_view(activations_matrices,window_shape=(2,2),axis=(2,3))
    slided_pool = slided_pool[:,:,::2,::2,:,:]
    max_pooled = np.max(slided_pool,axis=(4,5))
    return np.array(max_pooled)

def anti_max_pool_conv2(activations_matrices):

    stride_row,strice_col=(2,2)
    slided_pool = sliding_window_view(activations_matrices,window_shape=(2,2),axis=(2,3))
    slided_pool = slided_pool[:,:,::2,::2,:,:]
    max_pooled = np.max(slided_pool,axis=(4,5))
    max_indices_flat = np.argmax(slided_pool.reshape(32,64,5,5,4),axis=-1)
    offset_r = max_indices_flat // 2 
    offset_c = max_indices_flat % 2
    batch_idx,channel_idx,win_r,win_c = np.ogrid[:32,:64,:5,:5]
    target_rows = win_r*2 + offset_r
    target_cols = win_c*2+offset_c
    mask = np.zeros_like(activations_matrices,dtype=bool)
    mask[batch_idx,channel_idx,target_rows,target_cols]=True
    return activations_matrices*mask


def anti_max_pool_conv1(activations_matrices):

    stride_row,strice_col=(2,2)
    slided_pool = sliding_window_view(activations_matrices,window_shape=(2,2),axis=(2,3))
    slided_pool = slided_pool[:,:,::2,::2,:,:]
    max_pooled = np.max(slided_pool,axis=(4,5))
    max_indices_flat = np.argmax(slided_pool.reshape(32,32,13,13,4),axis=-1)
    offset_r = max_indices_flat // 2 
    offset_c = max_indices_flat % 2
    batch_idx,channel_idx,win_r,win_c = np.ogrid[:32,:32,:13,:13]
    target_rows = win_r*2 + offset_r
    target_cols = win_c*2+offset_c
    mask = np.zeros_like(activations_matrices,dtype=bool)
    mask[batch_idx,channel_idx,target_rows,target_cols]=True
    return activations_matrices*mask


In [11]:
def flatten_channel_last(activations):
    new_activations = np.array(activations).copy()
    t_new_activations = np.moveaxis(new_activations,1,-1)
    return new_activations.reshape(32,-1)

def unflatten_channel_last(flattened_gradients):
    return flattened_gradients.reshape(32,64,5,5)

In [12]:
def adam_optimizer():
    pass

In [13]:
def convolve(layer_type,filter_kernels,layer_biases,batch_inputs,stride=1):
    num_inputs = len(batch_inputs)
    num_filters,_,_,_ = filter_kernels.shape
    all_activations = []
    if layer_type == "conv1":
        windows = sliding_window_view(batch_inputs, window_shape=(1,3,3),axis=(1,2,3))
        all_activations = np.einsum("bDHWdhw,odhw->boHW",windows,filter_kernels)
    elif layer_type == "conv2":
        windows = sliding_window_view(batch_inputs, window_shape=(32,3,3),axis=(1,2,3))
        all_activations = np.einsum("bDHWdhw,odhw->boHW",windows,filter_kernels)
        
    return np.asarray(all_activations)

In [14]:
def dense_process(weights,biases,prev_activations):
    new_pre_activations = (prev_activations@weights)+biases
    return new_pre_activations

In [15]:
def forward_propagation(layer_types,layer_weights,layer_biases,batch_inputs):
    prev_activation = np.array(batch_inputs)
    preds=None
    global conv2_flattened_activatons
    for i,lt in enumerate(layer_types):
        if lt in ["conv1","conv2"]:
            pre_act = convolve(lt,layer_weights[i],layer_biases[i],prev_activation)
            pre_activations.append(pre_act)
            prev_activation = relu(pre_act)
            activations.append(prev_activation)
            max_pooled = max_pooling(prev_activation)
            prev_activation = max_pooled
            max_pooled_activations.append(max_pooled)
        else:
            if layer_types[i-1]=="conv2":
                prev_activation = flatten_channel_last(prev_activation)
                conv2_flattened_activatons = prev_activation
            if lt=="dense1":
                pre_act = dense_process(layer_weights[i],layer_biases[i],prev_activation)
                pre_activations.append(pre_act)
                prev_activation = relu(pre_act)
                activations.append(prev_activation)
            elif lt=="dense2":
                pre_act = dense_process(layer_weights[i],layer_biases[i],prev_activation)
                pre_activations.append(pre_act)
                preds = softmax(pre_act)
                activations.append(preds)
    
    return preds



In [20]:
def back_propagation(preds,layer_types,layer_weights,layer_biases,batch_true_labels):
    np_preds = np.asarray(preds)
    len_batch = len(preds)
    layer_errors = [None for i in range(len(layer_weights))]
    new_layer_weights = layer_weights.copy()
    new_layer_biases= layer_biases.copy()
    global activations
    for layer_index in range(len(layer_weights)-1,-1,-1):
        if layer_types[layer_index] == "dense2":
            delta_j = softmax_sparse_categorical_crossentropy_gradient(preds, batch_true_labels)
            weights_change = (activations[layer_index-1].T@delta_j)/len_batch
            new_layer_weights[layer_index] -= weights_change
            new_layer_biases[layer_index] -= np.sum(delta_j,axis=0)/len_batch
            layer_errors[layer_index] = delta_j
        elif layer_types[layer_index]=="dense1":
            curr_activations = activations[layer_index]
            delta_j = (layer_errors[layer_index+1] @ layer_weights[layer_index+1].T) * (curr_activations > 0).astype(float)
            layer_errors[layer_index] = delta_j
            weights_change = ((relu(conv2_flattened_activatons)).T @ delta_j)/len_batch
            new_layer_weights[layer_index] -= weights_change
            new_layer_biases[layer_index] -= np.sum(delta_j,axis=0,keepdims=True)/len_batch
        elif layer_types[layer_index]=="conv2":
            curr_activations = np.array(conv2_flattened_activatons)
            flattened_delta_j = (layer_errors[layer_index+1] @ layer_weights[layer_index+1].T) * (curr_activations > 0).astype(float)
            delta_j = unflatten_channel_last(flattened_delta_j)
            layers_pre_maxpool_activations = activations[layer_index].copy()
            max_pool_masked_activations = anti_max_pool_conv2(layers_pre_maxpool_activations)
            layer_errors[layer_index]=delta_j
            upscaled_delta_j = np.repeat(np.repeat(delta_j,2,axis=2),2,axis=3)
            upscaled_delta_j = np.pad(upscaled_delta_j, ((0, 0), (0, 0), (0, 1), (0, 1)), mode='constant')
            activations_matrix = max_pool_masked_activations * upscaled_delta_j
            prev_layer_activations_sliced = sliding_window_view(max_pooled_activations[layer_index-1],window_shape=(11,11),axis=(2,3))
            relu_derivative_activations = (activations_matrix * (activations_matrix > 0).astype(float))
            matrix_weights_change = np.einsum("bDHWhw,bohw->oDHW",prev_layer_activations_sliced,relu_derivative_activations)/len_batch
            new_layer_weights[layer_index] -= matrix_weights_change
            new_layer_biases[layer_index] -= np.einsum("ijkl->j",delta_j).reshape(1,64)/len_batch
        elif layer_types[layer_index]=="conv1":
            layers_pre_maxpool_activations = activations[layer_index].copy()
            max_pool_masked_activations = anti_max_pool_conv1(layers_pre_maxpool_activations)
            curr_weights = new_layer_weights[layer_index]
            delta_j = layer_errors[layer_index+1]
            rotated_weight_matrix = np.rot90(curr_weights,k=2)
            upscaled_delta_j = np.repeat(np.repeat(delta_j,2,axis=2),2,axis=3)
            upscaled_delta_j = np.pad(upscaled_delta_j, ((0, 0), (0, 0), (0, 1), (0, 1)), mode='constant')
            activations_matrix = max_pool_masked_activations * upscaled_delta_j
            prev_layer_activations_sliced = sliding_window_view(activations[layer_index-1],window_shape=activations_matrix.shape,axis=(1,2,3))
            matrix_weights_change = prev_layer_activations_sliced * (activations_matrix * (activations_matrix > 0).astype(float))
            new_layer_weights[layer_index] -= matrix_weights_change
            new_layer_biases[layer_index] -= delta_j
    return new_layer_weights, new_layer_biases




            
        

In [17]:
def xavier_initialization(layer_shapes, layer_weights=[],layer_biases=[]):
    rng = np.random.default_rng()
    for i in range(len(layer_shapes)):
        if len(layer_shapes[i]) <4:
            n_in,n_out = layer_shapes[i]
            biases = np.zeros((1,n_out))
        else:
            out_channels,in_channels,kw, kh = layer_shapes[i]
            receptive_field_size = kw*kh
            n_in = in_channels*receptive_field_size
            n_out = out_channels*receptive_field_size
            biases = np.zeros((1,out_channels))

        limit = np.sqrt(6/(n_in+n_out))
        weight = rng.uniform(-limit,limit,size=layer_shapes[i])
        layer_weights.append(weight)
        layer_biases.append(biases)
    return layer_weights,layer_biases


In [ ]:
def main_loop():
    layer_shapes = [(32,1,3,3),(64,32,3,3),(5*5*64,64),(64,26)]
    layer_types = ["conv1","conv2","dense1","dense2"]
    
    layer_weights = []
    layer_biases = []
    layer_weights,layer_biases = xavier_initialization(layer_shapes,layer_weights,layer_biases)
    
    epochs = 5
    batch_size = 32
    global X_train, y_train, max_pooled_activations, activations, pre_activations

    iters_per_epoch = int(math.ceil(len(X_train)/batch_size))
    for i in range(epochs):
        batch_iter = 0
        for j in range(iters_per_epoch):
            max_pooled_activations = []
            activations = []
            pre_activations = []
            batch_inputs = X_train[batch_iter:batch_iter+batch_size]
            batch_true_labels = y_train[batch_iter:batch_iter+batch_size]
            preds = forward_propagation(layer_types,layer_weights,layer_biases,batch_inputs)
            layer_weights, layer_biases = back_propagation(preds,layer_types,layer_weights,layer_biases,batch_true_labels)
            batch_iter+=batch_size

    weights_dict = {f"w_{i}":w for i,w in enumerate(layer_weights)}
    bias_dict = {f"b_{i}":b for i,b in enumerate(layer_biases)}
    np.savez("cnn_model.npz", **weights_dict, **bias_dict)
    
main_loop()

In [ ]:
def test_model():
    pass

In [ ]:
import numpy as np
import opt_einsum as oe
from numpy.lib.stride_tricks import sliding_window_view
np_arre = np.array([[[
    [(i*k)+l for i in range(j*3,(j+1)*3)] for j in range(3)
] for k in range(100,103)]for l in range(9)])
# print("np_arre\n",np_arre.shape,"\n", np_arre)
batch, depth, rows, columns = np_arre.shape
for i in range(batch):
    print(f"batch_{i}\n")
    print(np_arre[i,:,:,:])
windows = sliding_window_view(np_arre, window_shape=(2,2,2), axis=(1,2,3))
# print("windows\n",windows)
print("\n windows_shape \n", windows.shape)
# kernel = np.array([[[1,2],[3,4]],[[5,6],[7,8]]])
# path,info = oe.contract_path("bhwklc,klc->bhw", windows,kernel)
# print("path=> ", path)
# print("info=> ", info)
# output = np.einsum("ijmklp,klp->ijm", windows,kernel)
# print("output\n",output)

In [ ]:
print(np_arre[0,:,:,:])
print("\n=============\n")
print(windows[0,0,0,0,0,:,:])
#batch, depth_slide, height_slide, width_slide, slice_depth, slice_rows, slice_cols

In [ ]:
import numpy as np
import opt_einsum as oe
from numpy.lib.stride_tricks import sliding_window_view

batch_inputs = np.random.rand(32,28,28,1)
filters_shape = np.random.rand(32,3,3,1)

In [ ]:
windowd = sliding_window_view(batch_inputs, window_shape=(3,3,1),axis=(1,2,3))

In [ ]:
windowd.shape

In [ ]:
np.einsum("bDHWdhw,odhw->boHW",windowd,filters_shape)